In [8]:
import requests
import csv
import time
import socket
from datetime import datetime, timezone
from requests.exceptions import ReadTimeout, ConnectionError, RequestException

BASE_URL = "https://api.binance.com/api/v3/klines"
SYMBOL = "BTCUSDT"
INTERVAL = "1m"
LIMIT = 1000

# ===== DATE RANGE (DECEMBER 2025) =====
START_DATE = datetime(2025, 12, 1, tzinfo=timezone.utc)
END_DATE   = datetime(2025, 12, 31, 23, 59, tzinfo=timezone.utc)

START_TIME = int(START_DATE.timestamp() * 1000)
END_TIME   = int(END_DATE.timestamp() * 1000)

SLEEP_TIME = 0.25
MAX_RETRIES = 6


# ========== SAFE BINANCE REQUEST ==========
def safe_binance_get(params):
    for i in range(MAX_RETRIES):
        try:
            r = requests.get(BASE_URL, params=params, timeout=15)
            r.raise_for_status()
            return r.json()

        except socket.gaierror:
            time.sleep(2 ** i)

        except (ReadTimeout, ConnectionError):
            time.sleep(2 ** i)

        except RequestException:
            time.sleep(2 ** i)

    raise RuntimeError("Binance API unreachable after multiple retries")


# ========== DOWNLOAD + UTC CONVERSION ==========
with open("btc_usdt_1m_dec_2025.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "open_time",
        "open_time_utc",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "close_time_utc"
    ])

    current_time = START_TIME

    while current_time < END_TIME:
        params = {
            "symbol": SYMBOL,
            "interval": INTERVAL,
            "startTime": current_time,
            "limit": LIMIT
        }

        data = safe_binance_get(params)

        if not data:
            break

        for candle in data:
            open_time = candle[0]
            close_time = candle[6]

            if open_time > END_TIME:
                break

            # 🔹 Convert epoch ms → UTC
            open_time_utc = datetime.fromtimestamp(
                open_time / 1000, tz=timezone.utc
            )
            close_time_utc = datetime.fromtimestamp(
                close_time / 1000, tz=timezone.utc
            )

            writer.writerow([
                open_time,
                open_time_utc.strftime("%Y-%m-%d %H:%M:%S"),
                candle[1],  # open
                candle[2],  # high
                candle[3],  # low
                candle[4],  # close
                candle[5],  # volume
                close_time,
                close_time_utc.strftime("%Y-%m-%d %H:%M:%S")
            ])

        current_time = data[-1][6] + 1
        time.sleep(SLEEP_TIME)

print("✅ Done! BTCUSDT Dec-2025 data saved with UTC timestamps.")


✅ Done! BTCUSDT Dec-2025 data saved with UTC timestamps.


In [14]:
import requests
import csv
import time
import socket
from datetime import datetime, timezone
from requests.exceptions import ReadTimeout, ConnectionError, RequestException

BASE_URL = "https://api.binance.com/api/v3/klines"
SYMBOL = "BTCUSD"
INTERVAL = "1m"
LIMIT = 1000

# ===== DATE RANGE (DECEMBER 2025) =====
START_DATE = datetime(2026, 1, 1, tzinfo=timezone.utc)
END_DATE   = datetime(2026, 12, 31, 23, 59, tzinfo=timezone.utc)

START_TIME = int(START_DATE.timestamp() * 1000)
END_TIME   = int(END_DATE.timestamp() * 1000)

SLEEP_TIME = 0.25
MAX_RETRIES = 6


# ========== SAFE BINANCE REQUEST ==========
def safe_binance_get(params):
    for i in range(MAX_RETRIES):
        try:
            r = requests.get(BASE_URL, params=params, timeout=15)
            r.raise_for_status()
            return r.json()

        except socket.gaierror:
            time.sleep(2 ** i)

        except (ReadTimeout, ConnectionError):
            time.sleep(2 ** i)

        except RequestException:
            time.sleep(2 ** i)

    raise RuntimeError("Binance API unreachable after multiple retries")


# ========== DOWNLOAD + UTC CONVERSION ==========
with open("btc_usd_1m_jan_2025.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "open_time",
        "open_time_utc",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "close_time_utc"
    ])

    current_time = START_TIME

    while current_time < END_TIME:
        params = {
            "symbol": SYMBOL,
            "interval": INTERVAL,
            "startTime": current_time,
            "limit": LIMIT
        }

        data = safe_binance_get(params)

        if not data:
            break

        for candle in data:
            open_time = candle[0]
            close_time = candle[6]

            if open_time > END_TIME:
                break

            # 🔹 Convert epoch ms → UTC
            open_time_utc = datetime.fromtimestamp(
                open_time / 1000, tz=timezone.utc
            )
            close_time_utc = datetime.fromtimestamp(
                close_time / 1000, tz=timezone.utc
            )

            writer.writerow([
                open_time,
                open_time_utc.strftime("%Y-%m-%d %H:%M:%S"),
                candle[1],  # open
                candle[2],  # high
                candle[3],  # low
                candle[4],  # close
                candle[5],  # volume
                close_time,
                close_time_utc.strftime("%Y-%m-%d %H:%M:%S")
            ])

        current_time = data[-1][6] + 1
        time.sleep(SLEEP_TIME)

print("✅ Done! BTCUSDT Jan-2026 data saved with UTC timestamps.")


✅ Done! BTCUSDT Jan-2026 data saved with UTC timestamps.


In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone
import os

BASE_URL = "https://api.binance.com/api/v3/klines"
SYMBOL = "BTCUSDT"
INTERVAL = "1m"
LIMIT = 1000

OUTPUT_FOLDER = "data"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def datetime_to_milliseconds(dt):
    return int(dt.replace(tzinfo=timezone.utc).timestamp() * 1000)


def fetch_month_data(symbol, year, month):
    print(f"\nDownloading {symbol} - {year}-{month:02d}")

    start_dt = datetime(year, month, 1)
    
    # Next month start
    if month == 12:
        end_dt = datetime(year + 1, 1, 1)
    else:
        end_dt = datetime(year, month + 1, 1)

    start_time = datetime_to_milliseconds(start_dt)
    end_time = datetime_to_milliseconds(end_dt)

    all_rows = []

    while start_time < end_time:
        params = {
            "symbol": symbol,
            "interval": INTERVAL,
            "limit": LIMIT,
            "startTime": start_time,
            "endTime": end_time
        }

        response = requests.get(BASE_URL, params=params)

        if response.status_code != 200:
            print("Error:", response.text)
            break

        data = response.json()

        if not data:
            break

        all_rows.extend(data)

        # Move to next batch
        start_time = data[-1][0] + 1

        time.sleep(0.2)  # Safe delay

    print(f"Downloaded rows: {len(all_rows)}")
    return all_rows


def save_month(symbol, year, month):
    rows = fetch_month_data(symbol, year, month)

    if not rows:
        print("No data found.")
        return

    columns = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"
    ]

    df = pd.DataFrame(rows, columns=columns)

    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

    file_path = f"{OUTPUT_FOLDER}/{symbol}_{year}_{month:02d}_1m.parquet"
    df.to_parquet(file_path, index=False)

    print(f"Saved: {file_path}")


if __name__ == "__main__":
    
    # Example: Download Jan–March 2024
    months_to_download = [
        (2026, 2),
    ]

    for year, month in months_to_download:
        save_month(SYMBOL, year, month)


Downloaded rows: 34834
Saved: data/BTCUSDT_2026_02_1m.parquet


In [3]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone
import os

BASE_URL = "https://api.binance.com/api/v3/klines"
SYMBOL = "BTCUSDT"
INTERVAL = "1m"
LIMIT = 1000

OUTPUT_FOLDER = "data"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def datetime_to_milliseconds(dt):
    return int(dt.replace(tzinfo=timezone.utc).timestamp() * 1000)


def fetch_month_data(symbol, year, month):
    print(f"\nDownloading {symbol} - {year}-{month:02d}")

    start_dt = datetime(year, month, 1)

    # Calculate next month start
    if month == 12:
        end_dt = datetime(year + 1, 1, 1)
    else:
        end_dt = datetime(year, month + 1, 1)

    start_time = datetime_to_milliseconds(start_dt)
    end_time = datetime_to_milliseconds(end_dt)

    all_rows = []

    while start_time < end_time:
        params = {
            "symbol": symbol,
            "interval": INTERVAL,
            "limit": LIMIT,
            "startTime": start_time,
            "endTime": end_time
        }

        response = requests.get(BASE_URL, params=params)

        if response.status_code != 200:
            print("Error:", response.text)
            break

        data = response.json()

        if not data:
            break

        all_rows.extend(data)

        # Move forward
        start_time = data[-1][0] + 1

        time.sleep(0.2)  # Safe delay

    print(f"Downloaded rows: {len(all_rows)}")
    return all_rows


def save_month(symbol, year, month):
    rows = fetch_month_data(symbol, year, month)

    if not rows:
        print("No data found.")
        return

    columns = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"
    ]

    df = pd.DataFrame(rows, columns=columns)

    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

    file_path = f"{OUTPUT_FOLDER}/{symbol}_{year}_{month:02d}_1m.csv"
    df.to_csv(file_path, index=False)

    print(f"Saved: {file_path}")


if __name__ == "__main__":

    # Example: Download Jan–March 2024
    months_to_download = [
        (2026, 2)
    ]

    for year, month in months_to_download:
        save_month(SYMBOL, year, month)


Downloaded rows: 34836
Saved: data/BTCUSDT_2026_02_1m.csv


In [ ]:
import ccxt
import pandas as pd
from datetime import datetime, timedelta, timezone
import time

def fetch_crypto_data(exchange_name='binance', symbol='BTC/USDT', timeframe='1m', days_back=2):
    """
    Fetches historical minute-by-minute data for a specific crypto pair.
    """
    # Initialize the exchange
    exchange_class = getattr(ccxt, exchange_name)()
    
    # Calculate the timestamp for exactly 2 days ago (in milliseconds)
    now = datetime.now(timezone.utc)
    since = now - timedelta(days=days_back)
    since_timestamp = int(since.timestamp() * 1000)
    
    all_ohlcv = []
    
    print(f"Fetching {timeframe} data for {symbol} from {exchange_name.capitalize()}...")
    
    while True:
        try:
            # Fetch OHLCV (Open, High, Low, Close, Volume) data
            # Limit is usually 1000 for Binance, 720 for Kraken
            ohlcv = exchange_class.fetch_ohlcv(symbol, timeframe, since=since_timestamp, limit=1000)
            
            # If no more data is returned, we've caught up to the present
            if not ohlcv:
                break
                
            all_ohlcv.extend(ohlcv)
            
            # Update the 'since' timestamp to the last fetched candle's time + 1 minute (60,000 ms)
            # This ensures we don't fetch overlapping data in the next loop iteration
            last_timestamp = ohlcv[-1][0]
            since_timestamp = last_timestamp + 60000
            
            # Prevent hitting rate limits
            time.sleep(exchange_class.rateLimit / 1000)
            
        except Exception as e:
            print(f"An error occurred: {e}")
            break

    # Convert the raw list of lists into a Pandas DataFrame for easy analysis
    df = pd.DataFrame(all_ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    
    # Convert milliseconds timestamp to a readable human datetime format
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    
    return df

# --- Execution ---

# Note: Binance mostly trades in USDT (Tether) rather than native USD
df_bitcoin = fetch_crypto_data(exchange_name='binance', symbol='BTC/USDT', timeframe='1m', days_back=2)

print("\nData fetching complete!")
print(f"Total rows fetched: {len(df_bitcoin)}")

# Optional: Save to a CSV file
df_bitcoin.to_csv('btc_usdt_1m_data.csv', index=False)

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone
import os

BASE_URL = "https://api.binance.com/api/v3/klines"
SYMBOL = "ETHUSDT"
INTERVAL = "1m"
LIMIT = 1000

OUTPUT_FOLDER = "data"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def datetime_to_milliseconds(dt):
    return int(dt.replace(tzinfo=timezone.utc).timestamp() * 1000)


def fetch_month_data(symbol, year, month):
    print(f"\nDownloading {symbol} - {year}-{month:02d}")

    start_dt = datetime(year, month, 1)

    # Calculate next month start
    if month == 12:
        end_dt = datetime(year + 1, 1, 1)
    else:
        end_dt = datetime(year, month + 1, 1)

    start_time = datetime_to_milliseconds(start_dt)
    end_time = datetime_to_milliseconds(end_dt)

    all_rows = []

    while start_time < end_time:
        params = {
            "symbol": symbol,
            "interval": INTERVAL,
            "limit": LIMIT,
            "startTime": start_time,
            "endTime": end_time
        }

        response = requests.get(BASE_URL, params=params)

        if response.status_code != 200:
            print("Error:", response.text)
            break

        data = response.json()

        if not data:
            break

        all_rows.extend(data)

        # Move forward
        start_time = data[-1][0] + 1

        time.sleep(0.2)  # Safe delay

    print(f"Downloaded rows: {len(all_rows)}")
    return all_rows


def save_month(symbol, year, month):
    rows = fetch_month_data(symbol, year, month)

    if not rows:
        print("No data found.")
        return

    columns = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"
    ]

    df = pd.DataFrame(rows, columns=columns)

    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

    file_path = f"{OUTPUT_FOLDER}/{symbol}_{year}_{month:02d}_1m.csv"
    df.to_csv(file_path, index=False)

    print(f"Saved: {file_path}")


if __name__ == "__main__":

    # Example: Download Jan–March 2024
    months_to_download = [
        (2026, 2)
    ]

    for year, month in months_to_download:
        save_month(SYMBOL, year, month)


Downloaded rows: 40321
Saved: data/ETHUSDT_2026_02_1m.csv
